<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="300" alt="Skills Network Logo">
    </a>
</p>


# Evaluación y refinamiento de modelos


Tiempo estimado: **30** minutos
    

## Objetivos

Al terminar este laboratorio serás capaz de:

* Evaluar y refinar modelos de predicción
* Identificar sobreajuste (*overfitting*) y subajuste (*underfitting*) en los modelos
* Aplicar regresión Ridge en modelos con características polinómicas
* Usar la búsqueda en rejilla (*Grid Search*) para elegir los mejores hiperparámetros


<h2>Tabla de contenido</h2>
<ul>
    <li><a href="#ref1">Evaluación del modelado </a></li>
    <li><a href="#ref2">Sobreajuste, subajuste y selección de modelos </a></li>
    <li><a href="#ref3">Regresión Ridge </a></li>
    <li><a href="#ref4">Búsqueda en rejilla (Grid Search)</a></li>
</ul>


Este conjunto de datos estaba alojado en IBM Cloud Object Storage. Haz clic <a href="https://cocl.us/DA101EN_object_storage">AQUÍ</a> para obtener almacenamiento gratuito.


In [ ]:
# instalación de las versiones específicas que usa el laboratorio (solo Skills Network Labs)
#! mamba install pandas==1.3.3 -y
#! mamba install numpy=1.21.2 -y
#! mamba install sklearn=0.20.1 -y
#! mamba install   ipywidgets=7.4.2 -y

In [ ]:
# Instalación de las bibliotecas
# En el entorno local ya están instaladas; descomenta la línea si te falta alguna:
# %pip install pandas matplotlib scipy scikit-learn seaborn

In [ ]:
import pandas as pd
import numpy as np

# Importación de los datos ya depurados 
path = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DA0101EN-SkillsNetwork/labs/Data%20files/module_5_auto.csv'
df = pd.read_csv(path)

In [ ]:
# Guardamos una copia local del conjunto de datos (opcional)
df.to_csv('module_5_auto.csv')

 Primero usemos solo los datos numéricos:


In [ ]:
# Nos quedamos únicamente con las columnas numéricas (método privado de pandas)
df=df._get_numeric_data()
df.head()

<h2>Funciones para graficar</h2>
Primero importamos las bibliotecas seaborn y matplotlib para graficar.


In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

A continuación definimos las funciones que usaremos en el laboratorio para hacer gráficos en distintas etapas.


In [ ]:
def DistributionPlot(RedFunction, BlueFunction, RedName, BlueName, Title):
    width = 12
    height = 10
    plt.figure(figsize=(width, height))

    ax1 = sns.kdeplot(RedFunction, color="r", label=RedName)
    ax2 = sns.kdeplot(BlueFunction, color="b", label=BlueName, ax=ax1)

    plt.title(Title)
    plt.xlabel('Price (in dollars)')
    plt.ylabel('Proportion of Cars')
    plt.show()
    plt.close()

In [ ]:
def PollyPlot(xtrain, xtest, y_train, y_test, lr,poly_transform):
    width = 12
    height = 10
    plt.figure(figsize=(width, height))
    
    
    # datos de entrenamiento 
    # datos de prueba 
    # lr: objeto de regresión lineal 
    # poly_transform: objeto de transformación polinómica 
 
    xmax=max([xtrain.values.max(), xtest.values.max()])

    xmin=min([xtrain.values.min(), xtest.values.min()])

    x=np.arange(xmin, xmax, 0.1)


    plt.plot(xtrain, y_train, 'ro', label='Training Data')
    plt.plot(xtest, y_test, 'go', label='Test Data')
    plt.plot(x, lr.predict(poly_transform.fit_transform(x.reshape(-1, 1))), label='Predicted Function')
    plt.ylim([-10000, 60000])
    plt.ylabel('Price')
    plt.legend()

<h2 id="ref1">Parte 1: Entrenamiento y prueba</h2>

<p>Un paso importante al evaluar tu modelo es dividir los datos en entrenamiento y prueba. Colocaremos la variable objetivo <b>price</b> en un dataframe aparte, <b>y_data</b>:</p>


In [ ]:
# Variable objetivo: el precio de cada automóvil (en dólares)
y_data = df['price']

Quitamos el precio del dataframe y guardamos el resto en **x_data**:


In [ ]:
# Variables explicativas: todas las columnas menos el precio
x_data=df.drop('price',axis=1)

Ahora dividimos los datos aleatoriamente en entrenamiento y prueba con la función <b>train_test_split</b>. 


In [ ]:
from sklearn.model_selection import train_test_split


x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.10, random_state=1)


print("número de muestras de prueba :", x_test.shape[0])
print("número de muestras de entrenamiento:",x_train.shape[0])


El parámetro <b>test_size</b> fija la proporción de datos que se destina a prueba. Arriba, el conjunto de prueba es el 10 % del total. 


<div class="alert alert-danger alertdanger" style="margin-top: 20px">
<h1> Pregunta n.º 1:</h1>

<b>Usa la función "train_test_split" para dividir el conjunto de datos de modo que el 40 % de las muestras quede para prueba. Fija el parámetro "random_state" en cero. La salida de la función debe ser: "x_train1", "x_test1", "y_train1" y "y_test1".</b>
</div>


In [ ]:
# Escribe tu código a continuación y presiona Shift+Enter para ejecutar 


<details><summary>Haz clic aquí para ver la solución</summary>

```python
x_train1, x_test1, y_train1, y_test1 = train_test_split(x_data, y_data, test_size=0.4, random_state=0) 
print("número de muestras de prueba :", x_test1.shape[0])
print("número de muestras de entrenamiento:",x_train1.shape[0])
```

</details>


Importemos <b>LinearRegression</b> del módulo <b>linear_model</b>.


In [ ]:
from sklearn.linear_model import LinearRegression

 Creamos un objeto de regresión lineal:


In [ ]:
lre=LinearRegression()

Ajustamos el modelo usando la característica "horsepower":


In [ ]:
lre.fit(x_train[['horsepower']], y_train)

Calculemos el R^2 sobre los datos de prueba:


In [ ]:
lre.score(x_test[['horsepower']], y_test)

Observa que el R^2 es mucho menor en los datos de prueba que en los de entrenamiento.


In [ ]:
lre.score(x_train[['horsepower']], y_train)

<div class="alert alert-danger alertdanger" style="margin-top: 20px">
<h1> Pregunta n.º 2: </h1>
<b> 
Calcula el R^2 sobre los datos de prueba usando el 40 % del conjunto de datos para prueba.
</b>
</div>


In [ ]:
# Escribe tu código a continuación y presiona Shift+Enter para ejecutar 


<details><summary>Haz clic aquí para ver la solución</summary>

```python
x_train1, x_test1, y_train1, y_test1 = train_test_split(x_data, y_data, test_size=0.4, random_state=0)
lre.fit(x_train1[['horsepower']],y_train1)
lre.score(x_test1[['horsepower']],y_test1)

```

</details>


A veces no tienes suficientes datos de prueba; en ese caso conviene usar validación cruzada. Veamos varios métodos disponibles. 


<h2>Puntuación de validación cruzada</h2>


Importemos <b>cross_val_score</b> del módulo <b>model_selection</b>.


In [ ]:
from sklearn.model_selection import cross_val_score

Le pasamos el objeto, la característica ("horsepower") y los datos objetivo (y_data). El parámetro 'cv' indica el número de particiones (*folds*); aquí, 4. 


In [ ]:
Rcross = cross_val_score(lre, x_data[['horsepower']], y_data, cv=4)

La métrica por defecto es R^2. Cada elemento del arreglo tiene el R^2 promedio de su partición:


In [ ]:
Rcross

 Podemos calcular el promedio y la desviación estándar de nuestra estimación:


In [ ]:
print("El promedio de las particiones es", Rcross.mean(), "y la desviación estándar es" , Rcross.std())

Podemos usar el error cuadrático negativo como métrica si fijamos el parámetro 'scoring' en 'neg_mean_squared_error'. 


In [ ]:
-1 * cross_val_score(lre,x_data[['horsepower']], y_data,cv=4,scoring='neg_mean_squared_error')

<div class="alert alert-danger alertdanger" style="margin-top: 20px">
<h1> Pregunta n.º 3: </h1>
<b> 
Calcula el R^2 promedio usando dos particiones y luego el R^2 promedio de la segunda partición usando la característica "horsepower": 
</b>
</div>


In [ ]:
# Escribe tu código a continuación y presiona Shift+Enter para ejecutar 


<details><summary>Haz clic aquí para ver la solución</summary>

```python
Rc=cross_val_score(lre,x_data[['horsepower']], y_data,cv=2)
Rc.mean()

```

</details>


También puedes usar la función 'cross_val_predict' para predecir. Divide los datos en el número de particiones indicado: una queda para prueba y las demás para entrenamiento. Primero importamos la función:


In [ ]:
from sklearn.model_selection import cross_val_predict

Le pasamos el objeto, la característica <b>"horsepower"</b> y los datos objetivo <b>y_data</b>. El parámetro 'cv' indica el número de particiones (aquí, 4). Podemos ver una salida:


In [ ]:
yhat = cross_val_predict(lre,x_data[['horsepower']], y_data,cv=4)
yhat[0:5]

<h2 id="ref2">Parte 2: Sobreajuste, subajuste y selección de modelos</h2>

<p>Los datos de prueba (también llamados "out of sample" o fuera de muestra) miden mucho mejor cómo se comportará tu modelo en el mundo real. Una razón es el sobreajuste.

Veamos algunos ejemplos. Estas diferencias se notan más en la regresión lineal múltiple y en la regresión polinómica, así que exploraremos el sobreajuste en ese contexto.</p>


Creemos un modelo de regresión lineal múltiple y entrenémoslo usando <b>'horsepower'</b>, <b>'curb-weight'</b>, <b>'engine-size'</b> y <b>'highway-mpg'</b> como variables explicativas.


In [ ]:
lr = LinearRegression()
lr.fit(x_train[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']], y_train)

Predicción con los datos de entrenamiento:


In [ ]:
yhat_train = lr.predict(x_train[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']])
yhat_train[0:5]

Predicción con los datos de prueba: 


In [ ]:
yhat_test = lr.predict(x_test[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']])
yhat_test[0:5]

Evaluemos el modelo por separado con los datos de entrenamiento y con los de prueba. 


Examinemos la distribución de los valores predichos para los datos de entrenamiento.


In [ ]:
Title = 'Distribution  Plot of  Predicted Value Using Training Data vs Training Data Distribution'
DistributionPlot(y_train, yhat_train, "Actual Values (Train)", "Predicted Values (Train)", Title)

Figura 1: valores predichos con los datos de entrenamiento frente a los valores reales de entrenamiento. 


Hasta aquí el modelo parece aprender bien del conjunto de entrenamiento. ¿Qué ocurre cuando se enfrenta a datos nuevos del conjunto de prueba? Al generar valores con los datos de prueba, la distribución de lo predicho es muy distinta de la de los valores reales. 


In [ ]:
Title='Distribution  Plot of  Predicted Value Using Test Data vs Data Distribution of Test Data'
DistributionPlot(y_test,yhat_test,"Actual Values (Test)","Predicted Values (Test)",Title)

Figura 2: valores predichos con los datos de prueba frente a los valores reales de prueba. 


<p>Al comparar la figura 1 con la figura 2 se ve que la distribución de la figura 1 se ajusta mucho mejor a los datos. La diferencia en la figura 2 es evidente en el rango de 5 000 a 15 000, donde la forma de la distribución cambia por completo. Veamos si la regresión polinómica también pierde precisión al analizar el conjunto de prueba.</p>


In [ ]:
from sklearn.preprocessing import PolynomialFeatures

<h4>Sobreajuste</h4>
<p>El sobreajuste ocurre cuando el modelo se ajusta al ruido y no al proceso que genera los datos. Por eso, al probarlo con el conjunto de prueba rinde peor: está modelando ruido. Creemos un modelo polinómico de grado 5.</p>


Usemos el 55 % de los datos para entrenamiento y el resto para prueba:


In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.45, random_state=0)

Haremos una transformación polinómica de grado 5 sobre la característica <b>'horsepower'</b>. 


In [ ]:
pr = PolynomialFeatures(degree=5)
x_train_pr = pr.fit_transform(x_train[['horsepower']])
x_test_pr = pr.fit_transform(x_test[['horsepower']])
pr

Ahora creemos un modelo de regresión lineal llamado "poly" y entrenémoslo.


In [ ]:
poly = LinearRegression()
poly.fit(x_train_pr, y_train)

Podemos ver la salida del modelo con el método "predict". Guardamos los valores en "yhat".


In [ ]:
yhat = poly.predict(x_test_pr)
yhat[0:5]

Tomemos los cinco primeros valores predichos y comparémoslos con los valores reales. 


In [ ]:
print("Valores predichos:", yhat[0:4])
print("Valores reales:", y_test[0:4].values)

Usaremos la función "PollyPlot", definida al inicio del laboratorio, para mostrar los datos de entrenamiento, los de prueba y la función predicha.


In [ ]:
PollyPlot(x_train['horsepower'], x_test['horsepower'], y_train, y_test, poly,pr)

Figura 3: modelo de regresión polinómica. Los puntos rojos son datos de entrenamiento, los verdes datos de prueba y la línea azul es la predicción del modelo. 


La función estimada sigue los datos, pero alrededor de 200 hp empieza a separarse de los puntos. 


 R^2 de los datos de entrenamiento:


In [ ]:
poly.score(x_train_pr, y_train)

 R^2 de los datos de prueba:


In [ ]:
poly.score(x_test_pr, y_test)

El R^2 en entrenamiento es 0.5567, mientras que en prueba es -29.87. Cuanto menor el R^2, peor el modelo: un R^2 negativo es señal de sobreajuste.


Veamos cómo cambia el R^2 en los datos de prueba según el grado del polinomio y grafiquemos los resultados:


In [ ]:
Rsqu_test = []

order = [1, 2, 3, 4]
for n in order:
    pr = PolynomialFeatures(degree=n)
    
    x_train_pr = pr.fit_transform(x_train[['horsepower']])
    
    x_test_pr = pr.fit_transform(x_test[['horsepower']])    
    
    lr.fit(x_train_pr, y_train)
    
    Rsqu_test.append(lr.score(x_test_pr, y_test))

plt.plot(order, Rsqu_test)
plt.xlabel('order')
plt.ylabel('R^2')
plt.title('R^2 Using Test Data')
plt.text(3, 0.75, 'Maximum R^2 ')    

El R^2 aumenta poco a poco hasta el polinomio de grado 3; con el de grado 4 cae de forma drástica.


<div class="alert alert-danger alertdanger" style="margin-top: 20px">
<h1> Pregunta n.º 4a:</h1>

<b>Se pueden hacer transformaciones polinómicas con más de una característica. Crea un objeto "PolynomialFeatures" llamado "pr1" de grado dos.</b>
</div>


In [ ]:
# Escribe tu código a continuación y presiona Shift+Enter para ejecutar 


<details><summary>Haz clic aquí para ver la solución</summary>

```python
pr1=PolynomialFeatures(degree=2)

```

</details>


<div class="alert alert-danger alertdanger" style="margin-top: 20px">
<h1> Pregunta n.º 4b: </h1>

<b> 
 Transforma las muestras de entrenamiento y de prueba para las características 'horsepower', 'curb-weight', 'engine-size' y 'highway-mpg'. Pista: usa el método "fit_transform".</b>
</div>


In [ ]:
# Escribe tu código a continuación y presiona Shift+Enter para ejecutar 


<details><summary>Haz clic aquí para ver la solución</summary>

```python
x_train_pr1=pr1.fit_transform(x_train[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']])

x_test_pr1=pr1.fit_transform(x_test[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']])


```

</details>


<!-- La respuesta está abajo:

x_train_pr1=pr.fit_transform(x_train[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']])
x_test_pr1=pr.fit_transform(x_test[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']])

-->


<div class="alert alert-danger alertdanger" style="margin-top: 20px">
<h1> Pregunta n.º 4c: </h1>
<b> 
¿Cuántas dimensiones tiene la nueva característica? Pista: usa el atributo "shape".
</b>
</div>


In [ ]:
# Escribe tu código a continuación y presiona Shift+Enter para ejecutar 


<details><summary>Haz clic aquí para ver la solución</summary>

```python
x_train_pr1.shape # ahora hay 15 características


```

</details>


<div class="alert alert-danger alertdanger" style="margin-top: 20px">
<h1> Pregunta n.º 4d: </h1>

<b> 
Crea un modelo de regresión lineal "poly1" y entrénalo con el método "fit" usando las características polinómicas.</b>
</div>


In [ ]:
# Escribe tu código a continuación y presiona Shift+Enter para ejecutar 


<details><summary>Haz clic aquí para ver la solución</summary>

```python
poly1=LinearRegression().fit(x_train_pr1,y_train)


```

</details>


 <div class="alert alert-danger alertdanger" style="margin-top: 20px">
<h1> Pregunta n.º 4e: </h1>
<b>Usa el método "predict" para predecir con las características polinómicas y luego la función "DistributionPlot" para mostrar la distribución de lo predicho frente a los datos reales de prueba.</b>
</div>


In [ ]:
# Escribe tu código a continuación y presiona Shift+Enter para ejecutar 


<details><summary>Haz clic aquí para ver la solución</summary>

```python
yhat_test1=poly1.predict(x_test_pr1)

Title='Distribution  Plot of  Predicted Value Using Test Data vs Data Distribution of Test Data'

DistributionPlot(y_test, yhat_test1, "Actual Values (Test)", "Predicted Values (Test)", Title)

```

</details>


<div class="alert alert-danger alertdanger" style="margin-top: 20px">
<h1> Pregunta n.º 4f: </h1>

<b>Con el gráfico de distribución de arriba, describe (con palabras) las dos regiones donde los precios predichos son menos precisos respecto a los reales.</b>
</div>


In [ ]:
# Escribe tu código a continuación y presiona Shift+Enter para ejecutar 


<details><summary>Haz clic aquí para ver la solución</summary>

```python
# El valor predicho es mayor que el real en los autos del rango de 10 000 USD; al contrario, el precio predicho es menor que el real en el rango de 30 000 a 40 000 USD. Por eso el modelo es menos preciso en esos rangos.

```

</details>



<h2 id="ref3">Parte 3: Regresión Ridge</h2> 


 En esta sección repasamos la regresión Ridge y vemos cómo el parámetro alpha modifica el modelo. Nota: aquí los datos de prueba se usan como datos de validación.


 Hagamos una transformación polinómica de grado dos sobre los datos. 


In [ ]:
pr=PolynomialFeatures(degree=2)
x_train_pr=pr.fit_transform(x_train[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg','normalized-losses','symboling']])
x_test_pr=pr.fit_transform(x_test[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg','normalized-losses','symboling']])

 Importemos <b>Ridge</b> del módulo <b>linear models</b>.


In [ ]:
from sklearn.linear_model import Ridge

Creemos un objeto de regresión Ridge con el parámetro de regularización (alpha) igual a 1 


In [ ]:
RigeModel=Ridge(alpha=1)

Como en la regresión habitual, el modelo se entrena con el método <b>fit</b>.


In [ ]:
RigeModel.fit(x_train_pr, y_train)

 Del mismo modo puedes obtener una predicción: 


In [ ]:
yhat = RigeModel.predict(x_test_pr)

Comparemos las cuatro primeras predicciones con el conjunto de prueba: 


In [ ]:
print('predicho:', yhat[0:4])
print('conjunto de prueba:', y_test[0:4].values)

Elegimos el valor de alpha que minimiza el error de prueba. Para eso usamos un bucle for; además hay una barra de progreso que muestra cuántas iteraciones llevamos.


In [ ]:
from tqdm import tqdm

Rsqu_test = []
Rsqu_train = []
dummy1 = []
Alpha = 10 * np.array(range(0,1000))
pbar = tqdm(Alpha)

for alpha in pbar:
    RigeModel = Ridge(alpha=alpha) 
    RigeModel.fit(x_train_pr, y_train)
    test_score, train_score = RigeModel.score(x_test_pr, y_test), RigeModel.score(x_train_pr, y_train)
    
    pbar.set_postfix({"Puntuación de prueba": test_score, "Puntuación de entrenamiento": train_score})

    Rsqu_test.append(test_score)
    Rsqu_train.append(train_score)

Podemos graficar el valor de R^2 para distintos valores de alpha: 


In [ ]:
width = 12
height = 10
plt.figure(figsize=(width, height))

plt.plot(Alpha,Rsqu_test, label='validation data  ')
plt.plot(Alpha,Rsqu_train, 'r', label='training Data ')
plt.xlabel('alpha')
plt.ylabel('R^2')
plt.legend()

**Figura 4**: la línea azul es el R^2 de los datos de validación y la roja el R^2 de los datos de entrenamiento. El eje x representa los distintos valores de alpha. 


Aquí el modelo se construye y se prueba con los mismos datos, así que entrenamiento y prueba coinciden.

La línea roja de la figura 4 es el R^2 de los datos de entrenamiento: al aumentar alpha, el R^2 disminuye, es decir, el modelo rinde peor en entrenamiento.

La línea azul es el R^2 en los datos de validación: al aumentar alpha, el R^2 sube y luego se estabiliza.


<div class="alert alert-danger alertdanger" style="margin-top: 20px">
<h1> Pregunta n.º 5: </h1>

Haz una regresión Ridge. Calcula el R^2 usando las características polinómicas: entrena con los datos de entrenamiento y evalúa con los de prueba. Usa alpha = 10.
</div>


In [ ]:
# Escribe tu código a continuación y presiona Shift+Enter para ejecutar 


<details><summary>Haz clic aquí para ver la solución</summary>

```python
RigeModel = Ridge(alpha=10) 
RigeModel.fit(x_train_pr, y_train)
RigeModel.score(x_test_pr, y_test)

```

</details>



<h2 id="ref4">Parte 4: Búsqueda en rejilla (Grid Search)</h2>


El término alpha es un hiperparámetro. Sklearn incluye la clase <b>GridSearchCV</b>, que simplifica la búsqueda del mejor hiperparámetro.


Importemos <b>GridSearchCV</b> del módulo <b>model_selection</b>.


In [ ]:
from sklearn.model_selection import GridSearchCV

Creamos un diccionario con los valores del parámetro:


In [ ]:
parameters1= [{'alpha': [0.001,0.1,1, 10, 100, 1000, 10000, 100000, 100000]}]
parameters1

Crea un objeto de regresión Ridge:


In [ ]:
RR=Ridge()
RR

Crea el objeto de búsqueda en rejilla para Ridge:


In [ ]:
Grid1 = GridSearchCV(RR, parameters1,cv=4)

Para evitar un aviso de obsolescencia por el parámetro iid, en el laboratorio original se fija su valor en "None".

> **Nota (entorno local):** scikit-learn 1.4 o superior ya no acepta el parámetro `iid`. Si tu versión lanza el error `unexpected keyword argument 'iid'`, omítelo al crear `GridSearchCV`; el resultado de la búsqueda es el mismo.

Entrenamos el modelo:


In [ ]:
Grid1.fit(x_data[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']], y_data)

El objeto busca los mejores valores del parámetro en los datos de validación. Podemos obtener el estimador con los mejores parámetros y guardarlo en la variable BestRR así:


In [ ]:
BestRR=Grid1.best_estimator_
BestRR

 Ahora probamos el modelo con los datos de prueba:


In [ ]:
BestRR.score(x_test[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']], y_test)

<div class="alert alert-danger alertdanger" style="margin-top: 20px">
<h1> Pregunta n.º 6: </h1>
Haz una búsqueda en rejilla para el parámetro alpha y el de normalización, y encuentra los mejores valores:
</div>


In [ ]:
# Escribe tu código a continuación y presiona Shift+Enter para ejecutar 


<details><summary>Haz clic aquí para ver la solución</summary>

```python
parameters2 = [{'alpha': [0.001, 0.1, 1, 10, 100, 1000, 10000, 100000, 100000]}]

Grid2 = GridSearchCV(Ridge(), parameters2, cv=4)
Grid2.fit(x_data[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']], y_data)
best_alpha = Grid2.best_params_['alpha']
best_ridge_model = Ridge(alpha=best_alpha)
best_ridge_model.fit(x_data[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']], y_data)


```

</details>



### ¡Gracias por completar este laboratorio!


## Author

<a href="https://www.linkedin.com/in/joseph-s-50398b136/" target="_blank">Joseph Santarcangelo</a>


### Other Contributors

<a href="https://www.linkedin.com/in/mahdi-noorian-58219234/" target="_blank">Mahdi Noorian PhD</a>

Bahare Talayian

Eric Xiao

Steven Dong

Parizad

Hima Vasudevan

<a href="https://www.linkedin.com/in/fiorellawever/" target="_blank">Fiorella Wenver</a>

<a href=" https://www.linkedin.com/in/yi-leng-yao-84451275/ " target="_blank" >Yi Yao</a>.

<a href="https://www.coursera.org/instructor/~129186572/" targets="_blank" >Abhishek Gagneja</a>

## Change Log


|  Date (YYYY-MM-DD) |  Version | Changed By  |  Change Description |
|---|---|---|---|
| 2023-09-28 | 2.4| Abhishek Gagneja| Updated instructions |
| 2020-10-30  | 2.3  | Lakshmi  | Changed URL of csv              |
| 2020-10-05  | 2.2  | Lakshmi  | Removed unused library imports  |
| 2020-09-14  | 2.1  | Lakshmi  | Made changes in OverFitting section  |
| 2020-08-27  | 2.0  | Lavanya  |  Moved lab to course repo in GitLab  |


<hr>

## <h3 align="center"> © IBM Corporation 2023. All rights reserved. <h3/>
